In [2]:
import numpy as np
import time
from datetime import datetime
import json
import random

import torch
import torch.nn as nn
import torch.optim as optim

from torch_geometric.data import HeteroData
from torch_geometric.nn.conv.gcn_conv import gcn_norm
from torch_geometric.nn import global_mean_pool, global_max_pool

from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import Ridge

from tqdm import tqdm
import sys
from pyg_dataset import NetlistDataset

sys.path.append("models/layers/")
from models.model_att import GNN_node

In [3]:
dataset = torch.load("h_dataset.pt")
h_dataset = []
for data in dataset:
    h_dataset.append(data)

C:\Users\rebal\AppData\Local\Temp\ipykernel_1620\330159287.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  dataset = torch.load("h_dataset.pt")


In [4]:
load_data_indices = [idx for idx in range(len(h_dataset))]
all_train_indices = load_data_indices[:4] + load_data_indices[6:]
all_valid_indices = load_data_indices[4:5]
all_test_indices = load_data_indices[5:6]

In [14]:
X_train = []
y_train = []
for i in all_train_indices:
    X_train.append(np.array(h_dataset[i]['node']['x']))
    y_train.append(np.array(h_dataset[i]['variant_data_lst'][0][0]))

X_train = np.concatenate(X_train)
y_train = np.concatenate(y_train)

X_valid = []
y_valid = []
for i in all_valid_indices:
    X_valid.append(np.array(h_dataset[i]['node']['x']))
    y_valid.append(np.array(h_dataset[i]['variant_data_lst'][0][0]))

X_valid = np.concatenate(X_valid)
y_valid = np.concatenate(y_valid)

X_test = []
y_test = []
for i in all_test_indices:
    X_test.append(np.array(h_dataset[i]['node']['x']))
    y_test.append(np.array(h_dataset[i]['variant_data_lst'][0][0]))

X_test = np.concatenate(X_test)
y_test = np.concatenate(y_test)

mask = np.random.choice([True, False], size=y_train.shape[0], p=[0.1, 0.9])
X_train = X_train[mask]
y_train = y_train[mask]

In [8]:
X_train = X_train[: :-1]
X_test = X_test[:, :-1]

In [9]:
model = RandomForestRegressor(n_estimators=31, max_features=15, n_jobs=20)
model.fit(X_train, y_train)

RandomForestRegressor(max_features=15, n_estimators=31, n_jobs=20)

In [13]:
X_train.shape

(869677, 46)

In [15]:
preds_test = model.predict(X_test)
np.sqrt(mean_squared_error(preds_test, y_test))

11.082596296045208

In [31]:
preds = model.predict(X_valid)
np.sqrt(mean_squared_error(preds, y_valid))

11.452750095960738

In [30]:
preds_test = model.predict(X_test)
np.sqrt(mean_squared_error(preds_test, y_test))

5.914771284727459

In [23]:
y_valid

array([15.       , 15.833333 , 11.666667 , ...,  1.6666666,  2.5      ,
        0.8333333], dtype=float32)

In [24]:
preds

array([26.08730159, 26.19047625, 26.28571438, ...,  5.86507936,
        2.88095237,  2.93650793])